
# OSR 508GB — Stage‑2 Direct Miner v3.1

**直接 `Runtime → Run all`。不需要 Gemini。**

这不是把 508GB 再复制一份，也不是先造一个巨大 FTS 数据库。

它把 **Stage‑1 已验证的 1,788 个 immutable Parquet shards** 当作正文真源，然后增加一个可断点的研究扫描层：

- 读取 `STAGE1_VERIFIED.json` + `schema_footer_all_1788.csv`
- 自动识别 5 个真实 schema adapter
- 先对每种 schema 跑 smoke gate
- smoke 全过后自动继续全量 1,788 shards
- 同一个 **query pack** 内几十/几百个词一次扫描
- Aho–Corasick 多模式匹配；没有库时自动 fallback
- **不保存整库正文副本**
- 每个 shard 只落：
  - term row-hit count
  - occurrence count
  - term co-occurrence
  - bounded representative snippets
  - raw `source / shard / row_index / sha256` locator
- 每 shard 一个原子结果文件：断线后重新 Run all 自动跳过已完成项
- query pack 变化 → 自动进入新的 hash run directory，不污染旧结果
- 最后 reducer 生成全局 counts / co-occurrence / samples / status

> 默认内置一个上古史 / 神话 smoke query pack。以后只需改 Drive 中的
> `OSR_WORK_SPACE/Stage2_QueryPacks/active_query_pack.json`，
> engine 本身不用重写。

## v3.1 hardening

- fixed cross-filesystem checkpoint promotion (`/content` → Drive cannot use one-step `os.replace`)
- de-duplicates overlapping aliases for the same canonical entity
- single-character canonicals are opt-in only
- checkpoint verification now includes schema SHA + source bytes
- removed per-batch forced GC
- added raw-row rehydration + SHA verification helper


In [ ]:

# CELL 1 — mount + dependencies

from google.colab import drive
drive.mount("/content/drive")

!pip -q install pyahocorasick

from pathlib import Path
from collections import Counter, defaultdict
from itertools import combinations
import os, re, json, time, math, hashlib, random, gzip, gc, unicodedata, shutil
import pandas as pd
import numpy as np
import pyarrow as pa
import pyarrow.parquet as pq

try:
    import ahocorasick
    HAVE_AHO = True
except Exception:
    HAVE_AHO = False

ENGINE_VERSION = "OSR_508GB_STAGE2_DIRECT_MINER_V3_1"

print("✅ Environment ready")
print("Engine:", ENGINE_VERSION)
print("Aho-Corasick:", HAVE_AHO)


In [ ]:

# CELL 2 — paths + strict Stage-1 gate

WORK_ROOT = Path("/content/drive/MyDrive/OSR_WORK_SPACE")
STAGE1_ROOT = WORK_ROOT / "Stage1_Outputs_v2"
QUERY_PACK_ROOT = WORK_ROOT / "Stage2_QueryPacks"
RUNS_ROOT = WORK_ROOT / "Stage2_DirectMiner_v3_1"

STAGE1_STATUS = STAGE1_ROOT / "STAGE1_VERIFIED.json"
SCHEMA_CSV = STAGE1_ROOT / "schema_footer_all_1788.csv"
ACTIVE_QUERY_PACK = QUERY_PACK_ROOT / "active_query_pack.json"

for p in [STAGE1_STATUS, SCHEMA_CSV]:
    if not p.exists():
        raise FileNotFoundError(f"Missing required Stage-1 artifact: {p}")

stage1 = json.loads(STAGE1_STATUS.read_text(encoding="utf-8"))
schema_df = pd.read_csv(SCHEMA_CSV)

assert stage1.get("verified") is True
assert stage1.get("next_stage_allowed") is True
assert stage1["canonical_counts"]["Literature-zh"] == 233
assert stage1["canonical_counts"]["ChineseWebText2.0"] == 1555
assert stage1["canonical_counts"]["TOTAL"] == 1788
assert stage1["footer_schema_files_ok"] == 1788
assert stage1["footer_schema_files_error"] == 0
assert len(schema_df) == 1788
assert (schema_df["status"] == "OK").all()

QUERY_PACK_ROOT.mkdir(parents=True, exist_ok=True)
RUNS_ROOT.mkdir(parents=True, exist_ok=True)

EXPECTED_TOTAL_ROWS = int(schema_df["num_rows"].sum())
assert EXPECTED_TOTAL_ROWS == 202_581_374, EXPECTED_TOTAL_ROWS

print("✅ Stage-1 strict gate passed")
print("Canonical shards:", len(schema_df))
print("Total rows:", f"{int(schema_df['num_rows'].sum()):,}")


In [ ]:

# CELL 3 — create/read active query pack

DEFAULT_QUERY_PACK = {
    "name": "OSR_ancient_china_myth_seed_v1_1",
    "normalization": "NFC_CASEFOLD",
    "description": "Initial reusable research pack. Edit aliases/groups; engine remains unchanged.",
    "entities": [
        {"canonical": "黄帝", "group": "帝系", "aliases": ["黄帝", "黃帝", "轩辕", "軒轅"]},
        {"canonical": "炎帝", "group": "帝系", "aliases": ["炎帝"]},
        {"canonical": "神农", "group": "帝系", "aliases": ["神农", "神農"]},
        {"canonical": "伏羲", "group": "帝系", "aliases": ["伏羲", "伏犧", "庖牺", "庖犧"]},
        {"canonical": "女娲", "group": "帝系", "aliases": ["女娲", "女媧"]},
        {"canonical": "太昊", "group": "帝系", "aliases": ["太昊", "太皞"]},
        {"canonical": "少昊", "group": "帝系", "aliases": ["少昊", "少皞"]},
        {"canonical": "高阳", "group": "帝系", "aliases": ["高阳", "高陽"]},
        {"canonical": "高辛", "group": "帝系", "aliases": ["高辛"]},
        {"canonical": "共工", "group": "战争/水系", "aliases": ["共工"]},
        {"canonical": "相柳", "group": "战争/水系", "aliases": ["相柳", "相繇"]},
        {"canonical": "蚩尤", "group": "战争", "aliases": ["蚩尤"]},
        {"canonical": "鲧", "group": "治水", "aliases": ["鲧", "鯀"], "allow_single_char": True},
        {"canonical": "禹", "group": "治水", "aliases": ["大禹", "夏禹"], "allow_single_char": False},
        {"canonical": "祝融", "group": "神职", "aliases": ["祝融"]},
        {"canonical": "重黎", "group": "神职", "aliases": ["重黎", "重、黎", "重黎氏"]},
        {"canonical": "应龙", "group": "龙系", "aliases": ["应龙", "應龍"]},
        {"canonical": "烛龙", "group": "龙系", "aliases": ["烛龙", "燭龍", "烛阴", "燭陰"]},
        {"canonical": "不周山", "group": "地理", "aliases": ["不周山"]},
        {"canonical": "昆吾", "group": "氏族/方国", "aliases": ["昆吾"]},
        {"canonical": "有熊", "group": "氏族/方国", "aliases": ["有熊"]},
        {"canonical": "三苗", "group": "氏族/方国", "aliases": ["三苗", "苗民"]},
        {"canonical": "尊卢", "group": "古帝名", "aliases": ["尊卢", "尊盧"]},
        {"canonical": "赫胥", "group": "古帝名", "aliases": ["赫胥"]},
        {"canonical": "丽处", "group": "古帝名", "aliases": ["丽处", "麗處"]},
        {"canonical": "容成", "group": "古帝名", "aliases": ["容成", "容成氏"]},
        {"canonical": "大庭", "group": "古帝名", "aliases": ["大庭", "大庭氏"]},
        {"canonical": "博黄", "group": "古帝名", "aliases": ["博黄", "博黃"]},
        {"canonical": "奇肱", "group": "山海经", "aliases": ["奇肱", "奇肱国", "奇肱國"]},
        {"canonical": "九鼎", "group": "礼制/技术", "aliases": ["九鼎"]},
        {"canonical": "神鼎", "group": "礼制/技术", "aliases": ["神鼎"]},
        {"canonical": "鼎湖", "group": "礼制/地理", "aliases": ["鼎湖"]},
    ],
    "sampling": {
        "per_term_per_shard": 2,
        "global_per_term": 300,
        "multi_term_per_shard": 5,
        "snippet_radius_chars": 350
    }
}

if not ACTIVE_QUERY_PACK.exists():
    ACTIVE_QUERY_PACK.write_text(
        json.dumps(DEFAULT_QUERY_PACK, ensure_ascii=False, indent=2),
        encoding="utf-8"
    )
    print("✅ Created default active query pack:", ACTIVE_QUERY_PACK)
else:
    print("✅ Using existing active query pack:", ACTIVE_QUERY_PACK)

query_pack = json.loads(ACTIVE_QUERY_PACK.read_text(encoding="utf-8"))

def canonical_json(obj):
    return json.dumps(obj, ensure_ascii=False, sort_keys=True, separators=(",", ":"))

query_pack_sha256 = hashlib.sha256(
    canonical_json(query_pack).encode("utf-8")
).hexdigest()

RUN_ROOT = RUNS_ROOT / query_pack_sha256[:16]
SHARD_RESULT_DIR = RUN_ROOT / "shards"
FAILURE_DIR = RUN_ROOT / "failures"
SHARD_RESULT_DIR.mkdir(parents=True, exist_ok=True)
FAILURE_DIR.mkdir(parents=True, exist_ok=True)

(RUN_ROOT / "query_pack_snapshot.json").write_text(
    json.dumps(query_pack, ensure_ascii=False, indent=2),
    encoding="utf-8"
)

print("Query pack:", query_pack["name"])
print("Query pack SHA256:", query_pack_sha256)
print("Run root:", RUN_ROOT)


In [ ]:

# CELL 4 — validate query pack + build matcher

def norm_search(s):
    if s is None:
        return ""
    if not isinstance(s, str):
        s = str(s)
    s = s.replace("\r\n", "\n").replace("\r", "\n").replace("\x00", "")
    return unicodedata.normalize("NFC", s).casefold()

entities = query_pack.get("entities", [])
if not entities:
    raise RuntimeError("Query pack has no entities.")

pattern_to_canonical = {}
canonical_to_group = {}

for ent in entities:
    canonical = str(ent["canonical"]).strip()
    if not canonical:
        raise ValueError("Empty canonical entity.")

    canonical_to_group[canonical] = ent.get("group", "")
    allow_single = bool(ent.get("allow_single_char", False))
    aliases = list(ent.get("aliases", []))

    # Canonical form is auto-added only if it is not a one-character pattern,
    # unless the entity explicitly opts into single-char matching.
    candidate_aliases = list(aliases)
    if len(norm_search(canonical)) >= 2 or allow_single:
        candidate_aliases.append(canonical)

    for alias in dict.fromkeys(candidate_aliases):
        p = norm_search(alias).strip()
        if not p:
            continue
        if len(p) == 1 and not allow_single:
            print(f"⚠️ skipped single-char alias: {canonical} -> {alias}")
            continue
        if p in pattern_to_canonical and pattern_to_canonical[p] != canonical:
            raise ValueError(
                f"Alias collision after normalization: {alias!r} -> "
                f"{pattern_to_canonical[p]!r} and {canonical!r}"
            )
        pattern_to_canonical[p] = canonical

patterns = sorted(pattern_to_canonical.keys(), key=len, reverse=True)

if HAVE_AHO:
    automaton = ahocorasick.Automaton()
    for p in patterns:
        automaton.add_word(p, (p, pattern_to_canonical[p]))
    automaton.make_automaton()
else:
    fallback_regex = re.compile("|".join(re.escape(p) for p in patterns))

def raw_matches(text):
    s = norm_search(text)
    if HAVE_AHO:
        for end_idx, (pattern, canonical) in automaton.iter(s):
            start_idx = end_idx - len(pattern) + 1
            yield canonical, pattern, start_idx, end_idx + 1
    else:
        for m in fallback_regex.finditer(s):
            pattern = m.group(0)
            yield pattern_to_canonical[pattern], pattern, m.start(), m.end()

def dedupe_nested_matches(matches):
    # For the SAME canonical entity, overlapping aliases count once.
    # Example: 容成氏 should not count both 容成 and 容成氏.
    by_can = defaultdict(list)
    for can, pat, start, end in matches:
        by_can[can].append((pat, start, end))

    result = {}
    for can, ms in by_can.items():
        ms = sorted(ms, key=lambda x: (-(x[2] - x[1]), x[1], x[2]))
        kept = []
        for pat, start, end in ms:
            overlaps = any(not (end <= ks or start >= ke) for _, ks, ke in kept)
            if not overlaps:
                kept.append((pat, start, end))
        result[can] = sorted(kept, key=lambda x: x[1])
    return result

print("Canonical entities:", len(entities))
print("Matcher patterns:", len(patterns))
print("Matcher:", "Aho-Corasick" if HAVE_AHO else "regex fallback")


In [ ]:

# CELL 5 — schema audit + adapters

def parse_fields(s):
    return json.loads(s)

schema_df["fields"] = schema_df["schema_fields_json"].map(parse_fields)
schema_df["field_names"] = schema_df["fields"].map(
    lambda xs: [x["name"] for x in xs]
)

expected_schema_shapes = {
    ("text",),
    ("text", "score"),
    ("text", "title"),
    ("text", "source"),
    ("text", "domain", "toxicity", "quality_score"),
}

observed_shapes = {
    tuple(xs) for xs in schema_df["field_names"].tolist()
}

unknown = observed_shapes - expected_schema_shapes
if unknown:
    raise RuntimeError(f"Unexpected schema shape(s): {unknown}")

schema_summary = (
    schema_df.groupby(["source", "schema_sha256"], as_index=False)
    .agg(
        files=("path", "count"),
        rows=("num_rows", "sum"),
        bytes=("size_bytes", "sum"),
        field_names=("field_names", "first"),
    )
    .sort_values(["source", "files"], ascending=[True, False])
)

display(schema_summary)

# One representative per actual schema, choosing the smallest shard for smoke test.
rep_idx = schema_df.groupby(["source", "schema_sha256"])["size_bytes"].idxmin()
smoke_reps = (
    schema_df.loc[rep_idx]
    .sort_values(["source", "schema_sha256"])
    .reset_index(drop=True)
)

print("✅ All 5 observed schema shapes have explicit adapters.")
display(smoke_reps[["source", "filename", "size_bytes", "num_rows", "field_names"]])


In [ ]:

# CELL 6 — shard scanner

PER_TERM_PER_SHARD = int(query_pack["sampling"].get("per_term_per_shard", 2))
GLOBAL_PER_TERM = int(query_pack["sampling"].get("global_per_term", 300))
MULTI_TERM_PER_SHARD = int(query_pack["sampling"].get("multi_term_per_shard", 5))
SNIPPET_RADIUS = int(query_pack["sampling"].get("snippet_radius_chars", 350))

TARGET_BATCH_MIB = 12
MIN_BATCH_ROWS = 8
MAX_BATCH_ROWS = 1024

def safe_json(v):
    if v is None:
        return None
    if isinstance(v, (str, int, bool)):
        return v
    if isinstance(v, float):
        return None if (math.isnan(v) or math.isinf(v)) else v
    if isinstance(v, bytes):
        return {"__bytes_hex__": v.hex()}
    if isinstance(v, dict):
        return {str(k): safe_json(x) for k, x in v.items()}
    if isinstance(v, (list, tuple)):
        return [safe_json(x) for x in v]
    try:
        if pd.isna(v):
            return None
    except Exception:
        pass
    return str(v)

def sha256_text(s):
    if s is None:
        s = ""
    if not isinstance(s, str):
        s = str(s)
    return hashlib.sha256(s.encode("utf-8", errors="surrogatepass")).hexdigest()

def shard_key(row):
    payload = (
        f"{row['source']}|{row['filename']}|{row['schema_sha256']}|"
        f"{int(row['size_bytes'])}"
    )
    return hashlib.sha256(payload.encode("utf-8")).hexdigest()[:24]

def batch_rows_for(row):
    nrows = max(1, int(row["num_rows"]))
    avg_comp = max(1.0, float(row["size_bytes"]) / nrows)
    target = TARGET_BATCH_MIB * 1024 * 1024
    return max(MIN_BATCH_ROWS, min(MAX_BATCH_ROWS, int(target / avg_comp)))

def make_snippet(raw_text, search_start, search_end):
    raw = "" if raw_text is None else (
        raw_text if isinstance(raw_text, str) else str(raw_text)
    )
    a = max(0, min(len(raw), search_start) - SNIPPET_RADIUS)
    b = min(len(raw), max(0, min(len(raw), search_end)) + SNIPPET_RADIUS)
    return raw[a:b]

def reservoir_add(bucket, item, seen_count, k, rng):
    if len(bucket) < k:
        bucket.append(item)
    else:
        j = rng.randint(1, seen_count)
        if j <= k:
            bucket[j - 1] = item

def finish_result(
    row, rows_scanned, chars_scanned, null_text_rows, t0, batch_rows,
    row_hits, occurrence_hits, alias_occurrence_hits,
    cooccurrence, term_samples, multi_samples
):
    return {
        "engine_version": ENGINE_VERSION,
        "source": row["source"],
        "filename": row["filename"],
        "schema_sha256": row["schema_sha256"],
        "source_size_bytes": int(row["size_bytes"]),
        "expected_rows": int(row["num_rows"]),
        "rows_scanned": int(rows_scanned),
        "chars_scanned": int(chars_scanned),
        "null_text_rows": int(null_text_rows),
        "elapsed_seconds": float(time.time() - t0),
        "batch_rows": int(batch_rows),
        "row_hits": dict(row_hits),
        "occurrence_hits": dict(occurrence_hits),
        "alias_occurrence_hits": dict(alias_occurrence_hits),
        "cooccurrence": {"|||".join(k): v for k, v in cooccurrence.items()},
        "term_samples": dict(term_samples),
        "multi_term_samples": multi_samples,
    }

def scan_one_shard(row, sample_only_rows=None):
    path = Path(str(row["path"]))
    if not path.exists():
        raise FileNotFoundError(path)

    fields = list(row["field_names"])
    batch_rows = batch_rows_for(row)

    row_hits = Counter()
    occurrence_hits = Counter()
    alias_occurrence_hits = Counter()
    cooccurrence = Counter()

    term_samples = defaultdict(list)
    term_seen = Counter()
    multi_samples = []
    multi_seen = 0

    rows_scanned = 0
    chars_scanned = 0
    null_text_rows = 0

    rng = random.Random(
        int(hashlib.sha256(
            f"{query_pack_sha256}|{row['source']}|{row['filename']}".encode("utf-8")
        ).hexdigest()[:16], 16)
    )

    pf = pq.ParquetFile(path)
    t0 = time.time()

    for batch in pf.iter_batches(
        batch_size=batch_rows,
        columns=fields,
        use_threads=False,
    ):
        cols = {
            name: batch.column(i).to_pylist()
            for i, name in enumerate(fields)
        }

        for i in range(batch.num_rows):
            if sample_only_rows is not None and rows_scanned >= sample_only_rows:
                return finish_result(
                    row, rows_scanned, chars_scanned, null_text_rows, t0, batch_rows,
                    row_hits, occurrence_hits, alias_occurrence_hits,
                    cooccurrence, term_samples, multi_samples
                )

            row_index = rows_scanned
            raw = cols["text"][i]
            raw_text = "" if raw is None else (
                raw if isinstance(raw, str) else str(raw)
            )
            if raw is None:
                null_text_rows += 1

            rows_scanned += 1
            chars_scanned += len(raw_text)

            match_map = dedupe_nested_matches(raw_matches(raw_text))
            if not match_map:
                continue

            canonicals = sorted(match_map.keys())

            for canonical in canonicals:
                occurrences = match_map[canonical]
                row_hits[canonical] += 1
                occurrence_hits[canonical] += len(occurrences)

                for pat, _, _ in occurrences:
                    alias_occurrence_hits[pat] += 1

                pat, start, end = occurrences[0]
                metadata = {
                    name: safe_json(cols[name][i])
                    for name in fields if name != "text"
                }

                item = {
                    "canonical": canonical,
                    "group": canonical_to_group.get(canonical, ""),
                    "matched_pattern": pat,
                    "source": row["source"],
                    "shard": row["filename"],
                    "schema_sha256": row["schema_sha256"],
                    "row_index": int(row_index),
                    "row_id": hashlib.sha256(
                        f"{row['source']}|{row['filename']}|{row_index}".encode("utf-8")
                    ).hexdigest(),
                    "raw_text_sha256": sha256_text(raw_text),
                    "char_count": len(raw_text),
                    "metadata": metadata,
                    "snippet": make_snippet(raw_text, start, end),
                }

                term_seen[canonical] += 1
                reservoir_add(
                    term_samples[canonical],
                    item,
                    term_seen[canonical],
                    PER_TERM_PER_SHARD,
                    rng,
                )

            if 2 <= len(canonicals) <= 20:
                for a, b in combinations(canonicals, 2):
                    cooccurrence[(a, b)] += 1

                multi_seen += 1
                first = canonicals[0]
                _, start, end = match_map[first][0]
                multi_item = {
                    "canonicals": canonicals,
                    "source": row["source"],
                    "shard": row["filename"],
                    "schema_sha256": row["schema_sha256"],
                    "row_index": int(row_index),
                    "row_id": hashlib.sha256(
                        f"{row['source']}|{row['filename']}|{row_index}".encode("utf-8")
                    ).hexdigest(),
                    "raw_text_sha256": sha256_text(raw_text),
                    "snippet": make_snippet(raw_text, start, end),
                }
                reservoir_add(
                    multi_samples,
                    multi_item,
                    multi_seen,
                    MULTI_TERM_PER_SHARD,
                    rng,
                )

        del cols

    return finish_result(
        row, rows_scanned, chars_scanned, null_text_rows, t0, batch_rows,
        row_hits, occurrence_hits, alias_occurrence_hits,
        cooccurrence, term_samples, multi_samples
    )


In [ ]:

# CELL 7 — smoke gate: one representative shard per real schema

SMOKE_ROOT = RUN_ROOT / "_smoke"
SMOKE_ROOT.mkdir(parents=True, exist_ok=True)

SMOKE_ROWS_PER_SCHEMA = 64
smoke_results = []
smoke_failures = []

for i, row in smoke_reps.iterrows():
    print("\n" + "=" * 90)
    print(f"SMOKE [{i+1}/{len(smoke_reps)}] {row['source']} | {row['filename']}")
    try:
        result = scan_one_shard(row, sample_only_rows=SMOKE_ROWS_PER_SCHEMA)
        smoke_results.append(result)
        print(
            "✅",
            "rows =", result["rows_scanned"],
            "| chars =", f"{result['chars_scanned']:,}",
            "| batch =", result["batch_rows"],
            "| hits =", sum(result["row_hits"].values()),
        )
    except Exception as e:
        smoke_failures.append({
            "source": row["source"],
            "filename": row["filename"],
            "schema_sha256": row["schema_sha256"],
            "error": repr(e),
        })
        print("❌", repr(e))

(SMOKE_ROOT / "smoke_results.json").write_text(
    json.dumps(smoke_results, ensure_ascii=False, indent=2),
    encoding="utf-8",
)
(SMOKE_ROOT / "smoke_failures.json").write_text(
    json.dumps(smoke_failures, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

smoke_ok = (
    len(smoke_results) == 5
    and len(smoke_failures) == 0
    and all(x["rows_scanned"] > 0 for x in smoke_results)
)

print("\n" + "=" * 90)
print("SMOKE VERIFIED =", smoke_ok)
print("=" * 90)

if not smoke_ok:
    raise RuntimeError(
        "Smoke gate failed. Full 1,788-shard scan is blocked. "
        f"Inspect {SMOKE_ROOT / 'smoke_failures.json'}"
    )


In [ ]:

# CELL 8 — full 1,788-shard scan with hardened checkpoints

def result_path_for(row):
    return SHARD_RESULT_DIR / f"{shard_key(row)}.json.gz"

def failure_path_for(row):
    return FAILURE_DIR / f"{shard_key(row)}.json"

def atomic_write_gzip_json(obj, dst):
    # Cross-filesystem safe:
    # local write -> Drive temp copy -> same-filesystem atomic rename.
    local_tmp = Path("/content") / f"{dst.name}.{os.getpid()}.localtmp"
    drive_tmp = dst.with_name(dst.name + ".drivetmp")

    try:
        with gzip.open(local_tmp, "wt", encoding="utf-8") as f:
            json.dump(obj, f, ensure_ascii=False, separators=(",", ":"))

        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copyfile(local_tmp, drive_tmp)

        with gzip.open(drive_tmp, "rt", encoding="utf-8") as f:
            probe = json.load(f)

        if probe.get("status") != "OK":
            raise RuntimeError("Drive temp checkpoint verification failed")

        os.replace(drive_tmp, dst)  # same Drive filesystem
    finally:
        for p in (local_tmp, drive_tmp):
            try:
                if p.exists():
                    p.unlink()
            except Exception:
                pass

def valid_existing_result(row):
    p = result_path_for(row)
    if not p.exists():
        return False
    try:
        with gzip.open(p, "rt", encoding="utf-8") as f:
            obj = json.load(f)
        return (
            obj.get("status") == "OK"
            and obj.get("engine_version") == ENGINE_VERSION
            and obj.get("query_pack_sha256") == query_pack_sha256
            and obj.get("source") == row["source"]
            and obj.get("filename") == row["filename"]
            and obj.get("schema_sha256") == row["schema_sha256"]
            and int(obj.get("source_size_bytes", -1)) == int(row["size_bytes"])
            and int(obj.get("rows_scanned", -1)) == int(row["num_rows"])
        )
    except Exception:
        return False

SOURCE_ORDER = {"Literature-zh": 0, "ChineseWebText2.0": 1}
work_df = schema_df.copy()
work_df["_source_order"] = work_df["source"].map(SOURCE_ORDER)
work_df = work_df.sort_values(["_source_order", "filename"]).reset_index(drop=True)

completed_before = sum(
    valid_existing_result(row)
    for _, row in work_df.iterrows()
)
print("Already valid:", completed_before, "/", len(work_df))

for idx, row in work_df.iterrows():
    if valid_existing_result(row):
        if (idx + 1) % 50 == 0:
            print(f"[{idx+1}/1788] resumed/skipped")
        continue

    print(
        f"\n[{idx+1}/1788] {row['source']} | {row['filename']} "
        f"| rows={int(row['num_rows']):,}"
    )

    try:
        result = scan_one_shard(row)

        if result["rows_scanned"] != int(row["num_rows"]):
            raise RuntimeError(
                f"Row-count mismatch: {result['rows_scanned']} != {int(row['num_rows'])}"
            )

        result.update({
            "status": "OK",
            "query_pack_name": query_pack["name"],
            "query_pack_sha256": query_pack_sha256,
            "finished_at_unix": time.time(),
        })

        atomic_write_gzip_json(result, result_path_for(row))

        fp = failure_path_for(row)
        if fp.exists():
            fp.unlink()

        print(
            "✅ rows =", f"{result['rows_scanned']:,}",
            "| chars =", f"{result['chars_scanned']:,}",
            "| row_hits =", f"{sum(result['row_hits'].values()):,}",
            "| sec =", f"{result['elapsed_seconds']:.1f}",
        )

    except Exception as e:
        failure = {
            "status": "ERROR",
            "engine_version": ENGINE_VERSION,
            "query_pack_sha256": query_pack_sha256,
            "source": row["source"],
            "filename": row["filename"],
            "schema_sha256": row["schema_sha256"],
            "source_size_bytes": int(row["size_bytes"]),
            "path": row["path"],
            "error": repr(e),
            "failed_at_unix": time.time(),
        }
        failure_path_for(row).write_text(
            json.dumps(failure, ensure_ascii=False, indent=2),
            encoding="utf-8",
        )
        print("❌", repr(e))

    gc.collect()


In [ ]:

# CELL 9 — reducer: counts + co-occurrence + global reservoirs + verification

global_row_hits = Counter()
global_occurrence_hits = Counter()
global_alias_hits = Counter()
global_cooccurrence = Counter()
global_samples = defaultdict(list)
global_sample_seen = Counter()
multi_global = []
multi_seen = 0

ok_files = 0
bad_results = []
rng = random.Random(int(query_pack_sha256[:16], 16))

def global_reservoir_add(bucket, item, seen_count, k):
    if len(bucket) < k:
        bucket.append(item)
    else:
        j = rng.randint(1, seen_count)
        if j <= k:
            bucket[j - 1] = item

for _, row in work_df.iterrows():
    p = result_path_for(row)
    if not p.exists():
        bad_results.append({
            "source": row["source"],
            "filename": row["filename"],
            "reason": "MISSING_RESULT",
        })
        continue

    try:
        with gzip.open(p, "rt", encoding="utf-8") as f:
            obj = json.load(f)

        if not valid_existing_result(row):
            raise ValueError("Result verification mismatch")

        ok_files += 1
        global_row_hits.update(obj.get("row_hits", {}))
        global_occurrence_hits.update(obj.get("occurrence_hits", {}))
        global_alias_hits.update(obj.get("alias_occurrence_hits", {}))

        for k, v in obj.get("cooccurrence", {}).items():
            a, b = k.split("|||", 1)
            global_cooccurrence[(a, b)] += int(v)

        for canonical, samples in obj.get("term_samples", {}).items():
            for item in samples:
                global_sample_seen[canonical] += 1
                global_reservoir_add(
                    global_samples[canonical],
                    item,
                    global_sample_seen[canonical],
                    GLOBAL_PER_TERM,
                )

        for item in obj.get("multi_term_samples", []):
            multi_seen += 1
            global_reservoir_add(
                multi_global,
                item,
                multi_seen,
                min(1000, GLOBAL_PER_TERM * 2),
            )

    except Exception as e:
        bad_results.append({
            "source": row["source"],
            "filename": row["filename"],
            "reason": repr(e),
        })

term_rows = []
for ent in entities:
    canonical = ent["canonical"]
    term_rows.append({
        "canonical": canonical,
        "group": ent.get("group", ""),
        "row_hits": int(global_row_hits[canonical]),
        "occurrences": int(global_occurrence_hits[canonical]),
        "sample_count": len(global_samples.get(canonical, [])),
    })

term_df = pd.DataFrame(term_rows).sort_values(
    ["row_hits", "occurrences"], ascending=False
)

co_df = pd.DataFrame([
    {
        "entity_a": a,
        "entity_b": b,
        "cooccurring_rows": int(v),
    }
    for (a, b), v in global_cooccurrence.items()
]).sort_values("cooccurring_rows", ascending=False) if global_cooccurrence else pd.DataFrame(
    columns=["entity_a", "entity_b", "cooccurring_rows"]
)

alias_df = pd.DataFrame([
    {
        "normalized_alias": alias,
        "canonical": pattern_to_canonical.get(alias, ""),
        "occurrences": int(v),
    }
    for alias, v in global_alias_hits.items()
]).sort_values("occurrences", ascending=False) if global_alias_hits else pd.DataFrame(
    columns=["normalized_alias", "canonical", "occurrences"]
)

TERM_COUNTS_CSV = RUN_ROOT / "term_counts.csv"
CO_COUNTS_CSV = RUN_ROOT / "cooccurrence_counts.csv"
ALIAS_COUNTS_CSV = RUN_ROOT / "alias_counts.csv"
SAMPLES_JSONL = RUN_ROOT / "representative_samples.jsonl"
MULTI_JSONL = RUN_ROOT / "multi_term_samples.jsonl"
BAD_JSON = RUN_ROOT / "reducer_bad_results.json"
STATUS_JSON = RUN_ROOT / "STAGE2_DIRECT_MINER_VERIFIED.json"

term_df.to_csv(TERM_COUNTS_CSV, index=False)
co_df.to_csv(CO_COUNTS_CSV, index=False)
alias_df.to_csv(ALIAS_COUNTS_CSV, index=False)

with open(SAMPLES_JSONL, "w", encoding="utf-8") as f:
    for canonical in sorted(global_samples):
        for item in global_samples[canonical]:
            f.write(json.dumps(item, ensure_ascii=False) + "\n")

with open(MULTI_JSONL, "w", encoding="utf-8") as f:
    for item in multi_global:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

BAD_JSON.write_text(
    json.dumps(bad_results, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

verified = (
    smoke_ok
    and ok_files == 1788
    and len(bad_results) == 0
)

status = {
    "stage": ENGINE_VERSION,
    "verified": bool(verified),
    "query_pack_name": query_pack["name"],
    "query_pack_sha256": query_pack_sha256,
    "canonical_shards_expected": 1788,
    "canonical_shards_ok": int(ok_files),
    "canonical_shards_bad_or_missing": int(len(bad_results)),
    "source_rows_expected": EXPECTED_TOTAL_ROWS,
    "raw_corpus_immutable": True,
    "full_text_derivative_created": False,
    "outputs": {
        "term_counts": str(TERM_COUNTS_CSV),
        "alias_counts": str(ALIAS_COUNTS_CSV),
        "cooccurrence_counts": str(CO_COUNTS_CSV),
        "representative_samples": str(SAMPLES_JSONL),
        "multi_term_samples": str(MULTI_JSONL),
        "reducer_bad_results": str(BAD_JSON),
        "shard_result_dir": str(SHARD_RESULT_DIR),
        "query_pack_snapshot": str(RUN_ROOT / "query_pack_snapshot.json"),
    },
    "next_step": (
        "Use counts/samples to create targeted second-pass query packs or "
        "promote selected evidence into the research pipeline."
        if verified else
        "Rerun notebook; valid shards auto-skip and missing/failed shards retry."
    ),
}

STATUS_JSON.write_text(
    json.dumps(status, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

print("\n" + "=" * 90)
if verified:
    print("✅✅✅ STAGE‑2 DIRECT MINER VERIFIED = TRUE")
    print("All 1,788 canonical shards were scanned for this query pack.")
else:
    print("⚠️ STAGE‑2 not yet complete.")
    print("Valid shards:", ok_files, "/ 1788")
    print("Bad or missing:", len(bad_results))
    print("Re-run this notebook: completed shard results will be skipped.")
print("=" * 90)

print("\nTop term counts:")
display(term_df.head(30))

print("\nTop co-occurrences:")
display(co_df.head(30))

print("\nRun directory:")
print(RUN_ROOT)


In [ ]:

# CELL 10 — raw evidence rehydration + SHA verification
# Defines helpers only; does not automatically reread any large shard.

def resolve_shard(source, shard):
    m = schema_df[
        (schema_df["source"] == source) &
        (schema_df["filename"] == shard)
    ]
    if len(m) != 1:
        raise KeyError(f"Shard not uniquely resolved: {source} / {shard}")
    return m.iloc[0]

def fetch_raw_row(source, shard, row_index, expected_sha256=None):
    row = resolve_shard(source, shard)
    fields = list(row["field_names"])
    target = int(row_index)

    if target < 0 or target >= int(row["num_rows"]):
        raise IndexError(target)

    pf = pq.ParquetFile(str(row["path"]))
    offset = 0
    batch_size = batch_rows_for(row)

    for batch in pf.iter_batches(
        batch_size=batch_size,
        columns=fields,
        use_threads=False,
    ):
        if offset + batch.num_rows <= target:
            offset += batch.num_rows
            continue

        local_i = target - offset
        result = {
            name: batch.column(i)[local_i].as_py()
            for i, name in enumerate(fields)
        }

        raw_text = result.get("text") or ""
        actual_sha = sha256_text(raw_text)

        if expected_sha256 is not None and actual_sha != expected_sha256:
            raise RuntimeError(
                f"SHA mismatch: expected={expected_sha256} actual={actual_sha}"
            )

        return {
            "source": source,
            "shard": shard,
            "row_index": target,
            "schema_sha256": row["schema_sha256"],
            "raw_text_sha256": actual_sha,
            "row": result,
        }

    raise RuntimeError("Row not reached")

print("✅ fetch_raw_row() ready.")



## 正确的长期用法

**同一个 query pack 断线：**直接重新 `Run all`，验证过的 shard 自动跳过。

**换研究主题：**只改：

`MyDrive/OSR_WORK_SPACE/Stage2_QueryPacks/active_query_pack.json`

一次尽量把同一研究簇的正名、异名、简繁、古称、方国名、地名和神职名都塞进去，一次全库扫描同时拿到频率、共现和代表证据。

**sample 不是最终史料证据。** 要升级成 FACT / STRONG HYPOTHESIS 时，用 `fetch_raw_row(source, shard, row_index, expected_sha256)` 回读完整原 row，再进 PHW/HREX。

Stage‑3 永久全文索引只在真实使用证明“重复重扫成本 > 长期索引成本”时再建。
